In [1]:
import os
import sys
import torch 
import pandas as pd

from tqdm import tqdm
from transformers import AutoTokenizer
from transformers import DataCollatorWithPadding
from datasets import load_dataset, load_from_disk, Dataset

# Get the current working directory of the notebook
notebook_dir = os.getcwd()
# Add the parent directory to the system path
sys.path.append(os.path.join(notebook_dir, '../'))

# import log_files
from data_processing import DataProcessing
from text_generation_models import TextGenerationModelFactory

In [2]:
hf_api_key = os.getenv("HUGGING_FACE_TEXT_CLASSIFICATION_API_KEY")

## Load Data

In [3]:
# Get original dataset
base_data_path = DataProcessing.load_base_data_path(notebook_dir)
train_p_o_path = os.path.join(base_data_path, 'text_classification_tutorial/train_dict-v2.json')
test_p_o_path = os.path.join(base_data_path, 'text_classification_tutorial/test_dict-v2.json')
files_by_split = {
    'train': str(train_p_o_path),
    'test': str(test_p_o_path)
}
dataset = load_dataset('json', data_files=files_by_split)
dataset

DatasetDict({
    train: Dataset({
        features: ['Base Sentence', 'Ground Truth', 'label', '__index_level_0__'],
        num_rows: 4666
    })
    test: Dataset({
        features: ['Base Sentence', 'Ground Truth', 'label', '__index_level_0__'],
        num_rows: 1167
    })
})

In [4]:
dataset["train"]

Dataset({
    features: ['Base Sentence', 'Ground Truth', 'label', '__index_level_0__'],
    num_rows: 4666
})

## Load Model

1. `distilbert`: 
    - a compressed transformer-based model
    - derives from BERT, with
        - same encoder
        - fewer layers, so lighter
        - good performance
2. `base`: 
    - pretrained on general text using masked language modeling
    - no classification head
    - attach classification head with `AutoModelForSequenceClassification`

Thus, we are loading the `distilbert-base-uncased` model such that we can finetune it to our dataset.

In [30]:
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")
tokenizer

BertTokenizer(name_or_path='distilbert/distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [6]:
dataset['train']['Base Sentence']

Column(['Tommy Jr. estimates that on March 22, 2036, the win ratio for games he has will disimprove.', 'The American Heart Association predicts that on March 15, 2029, the prevalence of hypertension among adults at the state level could decrease.', 'The S&P 500 index increased in January 2022, according to Fidelity.', 'Dr. Liam Chen predicts that on 02/28/2029, the average household income in the United States will rise.', 'On 21/08/2024, the Mayo Clinic speculates that the fiber intake at suburban households in Canada may increase.', ...])

In [7]:
def tokenize_sentences(dataset_split, print_first_n=3):
    tokenized_sentences = []
    labels = []
    sentences = dataset_split['Base Sentence']
    dataset_labels = dataset_split['label']

    for idx in range(len(sentences)):
        sentence = sentences[idx]
        if sentence is not None:
            if idx < print_first_n:
                print(sentence)
            tokenized_sentence = tokenizer(sentence, truncation=True)
            tokenized_sentences.append(tokenized_sentence)
            labels.append(dataset_labels[idx])

    return tokenized_sentences, labels

tokenized_train_sentences, train_labels = tokenize_sentences(dataset['train'])
tokenized_test_sentences, test_labels = tokenize_sentences(dataset['test'])

Tommy Jr. estimates that on March 22, 2036, the win ratio for games he has will disimprove.
The American Heart Association predicts that on March 15, 2029, the prevalence of hypertension among adults at the state level could decrease.
The S&P 500 index increased in January 2022, according to Fidelity.
According to environmental scientist Dr. Sofia Rodriguez, the air quality index at Los Angeles would fall in August 2027.
According to Ms. Emily Lee, the graduation rates at Harvard University will fall in Q3 of 2027.
Coach Emily Chen predicts on 2024/08/21, the goal count at Manchester United will climb.


In [8]:
def convert_to_dataset(tokenized_sentences, labels):
    data = {
        "input_ids": [],
        "attention_mask": [],
        "label": []
    }

    for idx in range(len(tokenized_sentences)):
        data["input_ids"].append(tokenized_sentences[idx]["input_ids"])
        data["attention_mask"].append(tokenized_sentences[idx]["attention_mask"])
        data["label"].append(labels[idx])

    return Dataset.from_dict(data)

train_dataset = convert_to_dataset(tokenized_train_sentences, train_labels)
test_dataset = convert_to_dataset(tokenized_test_sentences, test_labels)

In [9]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [10]:
import evaluate

accuracy = evaluate.load("accuracy")

In [11]:
import numpy as np


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

In [12]:
id2label = {0: "NON-PREDICTION", 1: "PREDICTION"}
label2id = {"NON-PREDICTION": 0, "PREDICTION": 1}

In [13]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert/distilbert-base-uncased", num_labels=2, id2label=id2label, label2id=label2id
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
test_dataset.features

{'input_ids': List(Value('int32')),
 'attention_mask': List(Value('int8')),
 'label': Value('int64')}

In [15]:
len(train_dataset), len(test_dataset)

(4647, 1162)

In [19]:
output_dir = os.path.join(base_data_path, 'text_classification_tutorial', 'tutorial_my_data')
training_args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=.2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

/Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/.venv_predictions/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.530605,0.783133


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

('/Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/notebook_experiments/../data/text_classification_tutorial/tutorial_my_data/tokenizer_config.json',
 '/Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/notebook_experiments/../data/text_classification_tutorial/tutorial_my_data/tokenizer.json')

In [25]:
text = "This was a masterpiece. Not completely faithful to the books, but enthralling from beginning to end. Might be my favorite of the three."
text = "JPMorgan Chase forecasts that the net profit at Microsoft will decrease in 2024/08/21."
text = "They will win."

In [26]:
from transformers import pipeline

# model_path = os.path.join(data_path, '')

classifier = pipeline("text-classification", model=output_dir)
classifier(text)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'PREDICTION', 'score': 0.8311996459960938}]

In [27]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(output_dir)
inputs = tokenizer(text, return_tensors="pt")

In [28]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(output_dir)
with torch.no_grad():
    logits = model(**inputs).logits

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [29]:
predicted_class_id = logits.argmax().item()
model.config.id2label[predicted_class_id]

'PREDICTION'